# core

> Universal helper functions

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import pandas as pd
import polars as pl
import itertools

## column helpers

In [ ]:
#| export
def move_columns(
    df: pd.DataFrame,  # Input
    cols_to_move: str,  # Single 
    pos: int  # Target
) -> pd.DataFrame:
    """
    Move one or more columns to a specified position in a DataFrame.
    """

    if isinstance(cols_to_move, str):
        cols_to_move = [cols_to_move]

    cols = list(df.columns)

    for c in cols_to_move:
        cols.remove(c)

    new_cols = cols[:pos] + cols_to_move + cols[pos:]
    return df[new_cols]


In [ ]:
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6], 'C': [7, 8, 9]})
df

,A,B,C
0,1,4,7
1,2,5,8
2,3,6,9


In [ ]:
move_columns(df, 'C', 0)

,C,A,B
0,7,1,4
1,8,2,5
2,9,3,6


In [ ]:
sample_string = 'KnotenNr. der Stücklistenposition'

In [ ]:
sample_string = sample_string.lower()
sample_string

'knotennr. der stücklistenposition'

In [ ]:
[c for c in sample_string][:10]

['k', 'n', 'o', 't', 'e', 'n', 'n', 'r', '.', ' ']

In [ ]:
[c.isalnum() for c in sample_string][:10]

[True, True, True, True, True, True, True, True, False, False]

In [ ]:
[c if c.isalnum() else '_' for c in sample_string][:10]

['k', 'n', 'o', 't', 'e', 'n', 'n', 'r', '_', '_']

In [ ]:
sample_string = "".join([c if c.isalnum() else '_' for c in sample_string])
sample_string

'knotennr__der_stücklistenposition'

In [ ]:
sample_string.split('_')

['knotennr', '', 'der', 'stücklistenposition']

In [ ]:
[o for o in filter(None, sample_string.split('_'))]

['knotennr', 'der', 'stücklistenposition']

In [ ]:
'_'.join(filter(None,sample_string.split('_')))

'knotennr_der_stücklistenposition'

In [ ]:
#| export
def clean_string(input_string:str):
    """Cleans input_string"""
    processed_string = "".join(c if c.isalnum() else "_" for c in input_string.lower()).strip("_")
    return "_".join(filter(None, processed_string.split("_")))

In [ ]:
clean_string(sample_string)

'knotennr_der_stücklistenposition'

In [ ]:
#| hide
test_eq(clean_string('___'), '')
test_eq(clean_string('Mat# Desc.'), 'mat_desc')
test_eq(clean_string(''), '')
test_eq(clean_string('987654321'), '987654321')

In [ ]:
#| export
def clean_col_names(df):
    """Returns df with clean column names, supports both pandas and polars."""
    if isinstance(df, pd.DataFrame):
        df.columns = [clean_string(col) for col in df.columns]
        return df
    return df.rename({col: clean_string(col) for col in df.columns})

In [ ]:
#| hide
data = {
    "Cust_ID.": [101, 102, 103, 104, 105],
    "Order--Date": ["2024-02-01", "2024-02-02", "2024-02-03", "2024-02-04", "2024-02-05"],
    "Prdct.Name!": ["Widget A", "Widget B", "Gadget X", "Gadget Y", "Device Z"],
    "QTY___Ordered": [10, 20, 5, 7, 15],
    "Unit$Price": [99.99, 149.99, 249.99, 349.99, 199.99]
}

df = pd.DataFrame(data)

In [ ]:
list(clean_col_names(df).columns)

['cust_id', 'order_date', 'prdct_name', 'qty_ordered', 'unit_price']

In [ ]:
assert list(clean_col_names(df).columns) == ['cust_id','order_date','prdct_name','qty_ordered','unit_price']

In [ ]:
df = pl.DataFrame(data)
assert clean_col_names(df).columns == ['cust_id','order_date','prdct_name','qty_ordered','unit_price']

In [ ]:
df.head(2)

In [ ]:
df = clean_col_names(df)
df.head(2)

In [ ]:
#| export
def show_identical_columns(
    df: pd.DataFrame,  # The DataFrame to analyze
    columns: list      # The list of column names to compare
) -> pd.DataFrame:     # A DataFrame matrix showing identity status between columns
    "Checks if specified columns in `df` are identical."
    if not set(columns).issubset(df.columns):
        raise ValueError("One or more specified columns do not exist in the DataFrame.")

    column_matrix = pd.DataFrame(index=columns, columns=columns)
    for col_1, col_2 in itertools.combinations(columns, 2):
        column_matrix.loc[col_1, col_2] = all(df[col_1].eq(df[col_2]))

    return column_matrix

## etl functions

In [ ]:
#| export
def reduce_mem_usage(
    df: pd.DataFrame | pd.Series, # Input DataFrame or Series
    verbose: bool = True          # Whether to print memory usage reduction
    ) -> pd.DataFrame | pd.Series:    # Reduced DataFrame or Series
    """Reduces memory usage of a DataFrame or Series by downcasting numerical types."""
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']

    if isinstance(df, pd.DataFrame):
        start_mem = df.memory_usage().sum() / 1024**2
    else:  # it's a Series
        start_mem = df.memory_usage() / 1024**2

    for col in df.columns if isinstance(df, pd.DataFrame) else [df.name]:
        col_type = df[col].dtypes if isinstance(df, pd.DataFrame) else df.dtypes
        if col_type in numerics:
            c_min = df[col].min() if isinstance(df, pd.DataFrame) else df.min()
            c_max = df[col].max() if isinstance(df, pd.DataFrame) else df.max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8) if isinstance(df, pd.DataFrame) else df.astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16) if isinstance(df, pd.DataFrame) else df.astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32) if isinstance(df, pd.DataFrame) else df.astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64) if isinstance(df, pd.DataFrame) else df.astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16) if isinstance(df, pd.DataFrame) else df.astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32) if isinstance(df, pd.DataFrame) else df.astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64) if isinstance(df, pd.DataFrame) else df.astype(np.float64)

    if isinstance(df, pd.DataFrame):
        end_mem = df.memory_usage().sum() / 1024**2
    else:  # it's a Series
        end_mem = df.memory_usage() / 1024**2

    if verbose:
        print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))

    return df

In [ ]:
#| export
def group_resample(
    df: pd.DataFrame,      # Input DataFrame
    id_col: str,           # Column name to group/unstack by
    value_col: str,        # Column name containing values to aggregate
    date_col: str,         # Column name containing dates
    freq: str = "W",       # Resampling frequency (e.g., 'W', 'ME', 'D')
    aggfunc: str = "sum"   # Aggregation function (e.g., 'sum', 'mean')
    ) -> pd.DataFrame:         # Resampled and stacked DataFrame
    "Group, resample, and restack a DataFrame by ID and date."
    return (
        df.groupby([date_col, id_col])
        .agg({value_col: aggfunc})
        .unstack(id_col)
        .resample(freq)
        .sum()
        .stack(id_col)
        .reset_index()
    )

In [ ]:
#| export
def polars_resample(df, date_col='ds', group_cols='unique_id', agg_col='y', frequency='1mo'):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    df = df.with_columns(pl.col(date_col).dt.truncate(frequency))
    agg_df = df.group_by([date_col] + group_cols).agg(pl.col(agg_col).sum())

    dates = pl.date_range(
        agg_df[date_col].min(), agg_df[date_col].max(),
        interval=frequency, eager=True
    ).alias(date_col)

    groups = agg_df.select(group_cols).unique()
    grid = pl.DataFrame({date_col: dates}).join(groups, how='cross')

    return grid.join(agg_df, on=[date_col] + group_cols, how='left').with_columns(pl.col(agg_col).fill_null(0)).sort([*group_cols, date_col])

## validate functions

Tests to check if we use out of scope variables inside function

In [ ]:
a = 10

In [ ]:
def my_sum(b):
    return a + b

my_sum(32)

In [ ]:
import inspect

In [ ]:
# inspect.getclosurevars(eval("my_sum"))
inspect.getclosurevars(my_sum)

In [ ]:
def my_callback(result):
    print("Cell just finished running!")

get_ipython().events.register('post_run_cell', my_callback)

In [ ]:
get_ipython().events.unregister('post_run_cell', my_callback)

In [ ]:
import ast

In [ ]:
def check_funcs(result):
    tree = ast.parse(result.info.raw_cell)
    func_names = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
    if func_names:
        print(f"Functions defined: {func_names}")

In [ ]:
get_ipython().events.register('post_run_cell', check_funcs)

In [ ]:
get_ipython().events.unregister('post_run_cell', check_funcs)

In [ ]:
def check_funcs(result):
    tree = ast.parse(result.info.raw_cell)
    func_names = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
    if func_names:
        print(f"Functions defined: {func_names}")

In [ ]:
get_ipython().events.register('post_run_cell', check_funcs)

In [ ]:
get_ipython().events.unregister('post_run_cell', check_funcs)

In [ ]:
import ast
import inspect

In [ ]:
def check_global_deps(result):
    tree = ast.parse(result.info.raw_cell)
    func_names = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
    
    ns = get_ipython().user_ns
    for name in func_names:
        if name in ns:
            func = ns[name]
            cv = inspect.getclosurevars(func)
            if cv.globals:
                print(f"⚠️  '{name}' depends on global variables: {list(cv.globals.keys())}")

In [ ]:
get_ipython().events.register('post_run_cell', check_global_deps)

In [ ]:
def my_sum(b):
    return a + b

my_sum(32)

In [ ]:
get_ipython().events.unregister('post_run_cell', check_global_deps)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()